# CSE 151B Competition

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # run in only first time

# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment
# !~/.local/bin/uv venv .venv --seed

# # activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

# # Install dependencies — this is fast thanks to uv's parallel resolver
# # adding more constraints from piazza
# !~/.local/bin/uv pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# !~/.local/bin/uv pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter accelerate -c constraints.txt


# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [2]:
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"
DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/results.jsonl"
MAX_TOKENS  = 4096

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## I added for cuda verification
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")
##


import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
try:
    from vllm import LLM, SamplingParams
except ModuleNotFoundError:
    LLM = None
    SamplingParams = None
    print("vLLM is not installed; using Transformers backend instead.")
from tqdm import tqdm

True
1
NVIDIA L4


## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [3]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 943 questions  (300 MCQ, 643 free-form)

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [4]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Solve the problem step-by-step, showing all necessary work. "
    "Double-check your calculations before finalizing. "
    "Do not repeat a step you have already completed. "
    "Simplify all fractions and radicals. Do not use decimals unless the problem requires it. "
    "If the problem has no solution, write \\boxed{None}. "
    "If the problem is ambiguous, state your assumption in one sentence, then solve. "
    "Once you are confident in your final answer, write it inside \\boxed{}. "
    "Your final answer must appear exactly once inside \\boxed{}. "
    "Do not place intermediate results inside \\boxed{}. "
    "For multiple sub-answers, use a single \\boxed{} with comma separation, e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Step 1: Eliminate obviously wrong choices. "
    "Step 2: Solve the problem mathematically, showing only essential steps. "
    "Step 3: Match your result to the closest answer choice. "
    "You MUST select one letter. If your result does not exactly match any option, pick the closest one. "
    "Do not loop back or re-examine completed steps. "
    "Do not output 'none of the above' unless it is an explicit option. "
    "Output ONLY the letter inside \\boxed{}, e.g. \\boxed{C}. Nothing after the boxed answer."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().

Options:
A. Unchanged
B. Increased by ten percent
C. Reduced by one percent
D. Increased by  ...

── Free-form user prompt (first 200 chars) ──
Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]
b) $4 \cdot 3-2+2 \cdot 3=$ [ANS] ...



## 5. Load Model with vLLM (for general case, vLLM is faster)

We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [5]:
# !~/151B_SP26_Competition/.venv/bin/pip install "transformers>=4.51.0" --upgrade
# !~/151B_SP26_Competition/.venv/bin/pip install "vllm>=0.8.0" --upgrade

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

In [7]:
from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen3-4B-Thinking-2507",
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.90, # bump from 0.50 using more VRAM
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=64, # parallel sequences
    )

sampling_params = SamplingParams(
    max_tokens=4096, # reduce from 32768 unless we need full-length thinking, it COULD increase accuracy, but it would also take a while longer to generate.
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    n=3, # generate K self-consistency samples in ONE call
)

INFO 05-31 13:51:18 [utils.py:278] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 64, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 05-31 13:51:18 [model.py:617] Resolved architecture: Qwen3ForCausalLM
INFO 05-31 13:51:18 [model.py:1752] Using max model len 16384
INFO 05-31 13:51:18 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-31 13:51:20 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 05-31 13:51:20 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
WARNING 05-31 13:51:25 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#p

(EngineCore pid=60773) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=60773) INFO 05-31 13:51:42 [weight_utils.py:922] Filesystem type for checkpoints: EXT4. Checkpoint size: 7.49 GiB. Available RAM: 12.23 GiB.
(EngineCore pid=60773) INFO 05-31 13:51:42 [weight_utils.py:945] Auto-prefetch is disabled because the filesystem (EXT4) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
(EngineCore pid=60773) /home/jupyter/151B_SP26_Competition/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=60773)   torch._check_is_size(blocksize)
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:01<00:02,  1.22s/it]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:02<00:01,  1.28s/it]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:02<00:00,  1.33it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:02<00:00,  1.13it/s]
(EngineCore pid=60773) 


(EngineCore pid=60773) INFO 05-31 13:51:45 [model_runner.py:295] Model loading took 2.71 GiB and 6.493722 seconds
(EngineCore pid=60773) INFO 05-31 13:51:50 [backends.py:1089] Using cache directory: /home/jupyter/.cache/vllm/torch_compile_cache/bf16aad3e2/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=60773) INFO 05-31 13:51:50 [backends.py:1148] Dynamo bytecode transform time: 4.35 s
(EngineCore pid=60773) INFO 05-31 13:51:52 [backends.py:292] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 2.086 s
(EngineCore pid=60773) INFO 05-31 13:51:52 [decorators.py:311] Directly load AOT compilation from path /home/jupyter/.cache/vllm/torch_compile_cache/torch_aot_compile/073873b2a88e71858306ff49cac5af049bc58302ed6b1853139daa9e29ec0d04/rank_0_0/model
(EngineCore pid=60773) INFO 05-31 13:51:52 [monitor.py:53] torch.compile took 6.90 s in total
(EngineCore pid=60773) INFO 05-31 13:51:52 [monitor.py:81] Initial profiling/warmup run took 0.21 s
(Engi

Capturing CUDA graphs (PIECEWISE):  95%|█████████▍| 18/19 [00:02<00:00,  6.10it/s]/home/jupyter/151B_SP26_Competition/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
(EngineCore pid=60773)   torch._check_is_size(blocksize)
Capturing CUDA graphs (FULL): 100%|██████████| 11/11 [00:01<00:00,  6.54it/s]


(EngineCore pid=60773) INFO 05-31 13:52:00 [model_runner.py:661] Graph capturing finished in 5 secs, took 0.48 GiB
(EngineCore pid=60773) INFO 05-31 13:52:00 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=60773) INFO 05-31 13:52:00 [core.py:302] init engine (profile, create kv cache, warmup model) took 14.98 s (compilation: 6.90 s)
(EngineCore pid=60773) INFO 05-31 13:52:02 [vllm.py:977] Asynchronous scheduling is enabled.
(EngineCore pid=60773) INFO 05-31 13:52:02 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

### Generate with vLLM

In [ ]:
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

responses = [out.outputs[0].text.strip() for out in outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 943 questions...


Rendering prompts:   0%|          | 0/943 [00:00<?, ?it/s]

(EngineCore pid=60773) WARNING 05-31 13:52:44 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _topk_topp_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts:   0%|          | 0/2829 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
responses[0] #Just to check and quickly gauge what a full answer looks like

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = False # Set to False when running on the private test set, True for testing on public.

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

## Next Steps

This notebook gives you a working baseline. Here are directions to improve your score:

- **Prompt engineering** — try different system prompts or few-shot examples inside the user turn
- **Sampling parameters** — adjust `temperature`, `top_p`, or use majority voting across multiple samples
- **Fine-tuning** — the competition allows model fine-tuning; see the course resources for guidance

Good luck!

## Notes from Diego to the rest of the team

There's probably going to be a few errors that you'll have to go through to run this, above the VLLM loader, there's some commented out commands to download specific versions of VLLM and Transformer. I think those are the ones I'm on since they were the last ones I used, but I also went through a variety of other attempts to get it to work, if it doesn't, I guess google or ask an LLM to try to get a working combination of libaries with everything you need.

I did a handful of things to increase efficiency (i.e. more vram usage, lower token limit, more K self-consistency cells per call, trying to cut it off early, etc). For accuracy, I focussed a lot on the prompt engineering side of things. Since it's asked on the presentation, I originally had it reason a lot more, but that caused issues when experimenting with that. It had a tendency to be VERY redundant until it hit its token limit and got cut off without an answer. So, I tried to be a little careful, and make sure that it would keep things on the shorter end of things, especially for MCQs, since it LOVES to just go through every option, when some of the public questions it was tested on would have like 10 options.

To quickly describe how I got the environment going (even though I talked about it over text), I used Google's Agent Platform Workbench. Basically you can get 300 dollars in credits for free. You do need to put in your card details and then activate the paid thing though. To my understanding, it prioritizes the 300 dollars in credits, and won't automatically charge you or anything. Moving on though, you navigate over to the Agent Platform Workbench, you can't add a GPU JUST yet. You gotta search up the quota 'GPUS_ALL_REGIONS' and request it to be 1. For me, this took only a few minutes even it says it takes up to 2 business days. FINALLY, you can then set up the environment. Go to the bottom, over to advanced, then you can go to hardware, into GPU, and select the L4 GPU. T4 is slightly cheaper, but notably worse. I think H100 is also offered, but its like 70 dollars an hour or something crazy like that. The environment with an L4 is like 0.50 an hour. It takes like 10 minutes for the environment to get set up, but once you're in, its literally the same ui as datahub. So just go through it like you normally would.

I'm going to be busy for most of Sunday, so unfortunately the remaining stuff (presentation and improving the accuracy further) is probably going to be up to you guys, but I think this is at least a pretty solid starting point, considering how problematic the environments have been for us. I'm also working on an initial kaggle submission, since we have up to 5 a day, and getting one down is just a good idea. I got 10 prompts answered in roughly 5 minutes, I haven't tested it with larger amounts at the time of writing this, since I'm improving it on batches of 10 in the public set before I commit to the full 1000 in the private set.

To test it on the public set, change it near the top (over where the cell with the GPU stuff is), set SAVE_EVAL to True in #9, and change "for item in data:" to "for item in data[:10]:" in #6.

ALSO, IMPORTANT, WHEN YOU GENERATE NEW RESULTS, YOU NEED TO HAVE THE RESULTS FOLDER EMPTY. It doesn't replace the file, so if there's one already in there, you'll still just have the old one.